# Clustering for spatial hotspot detection: Uber NYC, Ecoli

K-Means (random init), K-Means++, DBSCAN and HDBSCAN on Uber pickups in New York (April 2014; Kaggle)
and Ecoli protein localisation (UCI, labelled). Run the sections in order; each writes its table to `results/hotspot/`
as soon as it is complete.

1. Data loading
2. Parameter sweeps: K-Means k, DBSCAN and HDBSCAN by the shared rule
3. Main comparison at the chosen settings
4. Ablation A: initialisation (random vs K-Means++, 30 seeds)
5. Ablation B: distance metric on coordinates
6. Ablation C: feature scaling (Ecoli)
7. Ablation D: dimensionality (Ecoli, PCA)
8. Fake-data check
9. Shared-table rows for Uber and Ecoli (report, Section 3.3)
10. Case statistics behind the Uber (D3) and Ecoli (D4) paragraphs
11. Figures

In [ ]:
import json
import time
import warnings

import numpy as np
import pandas as pd
from scipy.stats import spearmanr
from sklearn.decomposition import PCA

from sc4020 import methods, tuning
from sc4020.datasets import load_dataset
from sc4020.datasets.preprocess_dataset import EARTH_RADIUS_KM
from sc4020.metrics import NOISE_LABEL, evaluate, pairwise_ari
from sc4020.paths import results_dir
from sc4020.tuning import null_gap

warnings.filterwarnings("ignore")

In [ ]:
SEED = 42
TOPIC = "hotspot"
K_GRID = list(range(2, 26))
N_SEEDS = 30
N_FAKE = 3
SILHOUETTE_SAMPLE = 5000  # the K rule estimates silhouettes on a sample for large data
KMEANS_INITS = {"K-Means (random init)": "random", "K-Means": "k-means++"}
# Grids from the report (Section 3.2, "Parameter grids"): eps at 0.1 to 3 times the k-distance knee.
EPS_FACTORS = (0.10, 0.15, 0.22, 0.32, 0.45, 0.6, 0.8, 1.0, 1.3, 1.7, 2.2, 3.0)
COLUMNS = [
    "n_clusters",
    "noise_fraction",
    "largest_cluster_share",
    "silhouette",
    "davies_bouldin",
    "ari",
    "nmi",
]
AIRPORTS = {"JFK": (40.645, -73.780), "LaGuardia": (40.774, -73.872)}
MIDTOWN_BOX = {
    "lat": (40.70, 40.77),
    "lon": (-74.02, -73.96),
}  # lower and midtown Manhattan

out = results_dir(TOPIC)

Helpers: the K-Means k sweep, the density sweeps and one row of scores per fit.

In [ ]:
def hdbscan_sizes(n):
    if n > 5000:
        return [10, 15, 25, 40, 60, 100, 150, 250]
    return [10, 15, 25, 40, 60, 100] if n > 1000 else [5, 8, 12, 20, 30]


def is_spatial(ds) -> bool:
    return "coords_deg" in ds


def eps_in_km(ds, eps):
    """DBSCAN eps in km: Uber's coordinates are already projected to km."""
    return float(eps)


def record(store, ds, method, params, labels, seconds, X=None, extra=None):
    row = {
        "dataset": ds.name,
        "method": method,
        "params": json.dumps(params, sort_keys=True),
        "fit_seconds": seconds,
        **evaluate(ds.data if X is None else X, labels, ds.target, seed=SEED),
        **(extra or {}),
    }
    store.append(row)
    return row


def k_sweep(ds, sweep):
    inertias = []
    for k in K_GRID:
        model = methods.kmeans(k, random_state=SEED)
        labels, seconds = methods.fit_predict_timed(model, ds.data)
        inertias.append(float(model.inertia_))
        record(
            sweep,
            ds,
            "K-Means",
            {"n_clusters": k},
            labels,
            seconds,
            extra={"inertia": float(model.inertia_), "n_iter": int(model.n_iter_)},
        )
    rows = [r for r in sweep if r["dataset"] == ds.name and r["method"] == "K-Means"]
    k_silhouette = K_GRID[int(np.nanargmax([r["silhouette"] for r in rows]))]
    k_null_gap = tuning.select_kmeans_k_by_null_gap(
        ds.data,
        k_range=K_GRID,
        sample_size=SILHOUETTE_SAMPLE if len(ds.data) > SILHOUETTE_SAMPLE else None,
        random_state=SEED,
    ).params["n_clusters"]
    choice = {
        "elbow": int(tuning.k_at_inertia_elbow(inertias, K_GRID)),
        "silhouette": int(k_silhouette),
        "null_gap": int(k_null_gap),
        # the shared rule: the number of classes when known, else the k with the largest gap
        # over fake data among seed-stable k (tuning.select_kmeans_k_by_null_gap)
        "selected": int(ds.n_true_clusters or k_null_gap),
    }
    if ds.target is not None:
        choice["best_ari"] = int(K_GRID[int(np.nanargmax([r["ari"] for r in rows]))])
    return choice


def density_sweep(ds, sweep, method, selection):
    for t in selection.tried:
        keys = (
            ("eps", "min_samples")
            if method == "DBSCAN"
            else ("min_cluster_size", "min_samples")
        )
        params = {k: t[k] for k in keys}
        labels = (
            methods.dbscan(**params)
            if method == "DBSCAN"
            else methods.hdbscan(**params)
        ).fit_predict(ds.data)
        record(sweep, ds, method, params, labels, np.nan)


def fitter(method, p):
    if method == "K-Means (random init)":
        return lambda Z: methods.kmeans(
            p["n_clusters"], init="random", random_state=SEED
        ).fit_predict(Z)
    if method == "K-Means":
        return lambda Z: methods.kmeans(p["n_clusters"], random_state=SEED).fit_predict(
            Z
        )
    if method == "DBSCAN":
        return lambda Z: methods.dbscan(**p).fit_predict(Z)
    return lambda Z: methods.hdbscan(**p).fit_predict(Z)

## 1. Data loading

In [ ]:
datasets = [
    load_dataset("uber_pickups"),
    load_dataset("ecoli"),
]
for ds in datasets:
    print(f"{ds.name:22s} n={len(ds.data):6d} d={ds.data.shape[1]}")

## 2. Parameter sweeps

K-Means over k = 2..25 (elbow, silhouette, gap over fake data), and the DBSCAN and HDBSCAN grids,
with eps at 0.1 to 3 times the k-distance knee. Writes `sweep.csv`, `chosen_params.json` and
`kdist_curves.json`.

In [ ]:
sweep, main_rows, ablation, seed_rows, fake_rows = [], [], [], [], []
chosen = {"k": {}, "dbscan": {}, "hdbscan": {}, "kdist": {}}
curves = {}

for ds in datasets:
    X, y = ds.data, ds.target
    chosen["k"][ds.name] = k_sweep(ds, sweep)
    base_ms = tuning.min_samples_from_dimension(X.shape[1])
    curve = tuning.k_distance_curve(X, base_ms)
    curves[ds.name] = curve.tolist()
    chosen["kdist"][ds.name] = {
        "min_samples": base_ms,
        "knee": tuning.find_knee(curve)[1],
    }
    big = len(X) > 1000
    grids = {
        "dbscan": lambda: tuning.select_dbscan_by_shared_rule(
            X,
            min_samples_grid=[4, 8, 16, 32] if big else [3, 5, 8, 12],
            eps_factors=EPS_FACTORS,
            y_true=y,
        ),
        "hdbscan": lambda: tuning.select_hdbscan_by_shared_rule(
            X,
            min_cluster_sizes=hdbscan_sizes(len(X)),
            min_samples_grid=(None,),
            y_true=y,
        ),
    }
    for method, key in [("DBSCAN", "dbscan"), ("HDBSCAN", "hdbscan")]:
        selection = grids[key]()
        chosen[key][ds.name] = {
            "rule": selection.params,
            "valid": selection.valid,
            "best_ari": selection.best_ari,
        }
        density_sweep(ds, sweep, method, selection)
    print(
        f"  {ds.name:22s} k={chosen['k'][ds.name]}  DBSCAN={chosen['dbscan'][ds.name]['rule']}  "
        f"HDBSCAN={chosen['hdbscan'][ds.name]['rule']}",
        flush=True,
    )

pd.DataFrame(sweep).to_csv(out / "sweep.csv", index=False)
(out / "chosen_params.json").write_text(json.dumps(chosen, indent=2, default=str))
(out / "kdist_curves.json").write_text(json.dumps(curves))
pd.DataFrame(chosen["k"]).T

## 3. Main comparison at the chosen settings

Writes `main.csv`.

In [ ]:
for ds in datasets:
    k = chosen["k"][ds.name]["selected"]
    for name, init in KMEANS_INITS.items():
        labels, seconds = methods.fit_predict_timed(
            methods.kmeans(k, init=init, random_state=SEED), ds.data
        )
        record(main_rows, ds, name, {"n_clusters": k}, labels, seconds)
    for method, factory, key in [
        ("DBSCAN", methods.dbscan, "dbscan"),
        ("HDBSCAN", methods.hdbscan, "hdbscan"),
    ]:
        params = chosen[key][ds.name]["rule"]
        labels, seconds = methods.fit_predict_timed(factory(**params), ds.data)
        record(main_rows, ds, method, params, labels, seconds)

pd.DataFrame(main_rows).to_csv(out / "main.csv", index=False)
pd.DataFrame(main_rows)[
    ["dataset", "method", "params", "n_clusters", "noise_fraction", "silhouette", "ari"]
].round(3)

## 4. Ablation A: initialisation

30 seeds with one start each, random vs K-Means++. Writes `init_seeds.csv`.

In [ ]:
for ds in datasets:
    k = chosen["k"][ds.name]["selected"]
    for name, init in KMEANS_INITS.items():
        runs = []
        for seed in range(N_SEEDS):
            model = methods.kmeans(k, init=init, n_init=1, random_state=seed)
            labels, seconds = methods.fit_predict_timed(model, ds.data)
            ari = (
                evaluate(ds.data, labels, ds.target)["ari"]
                if ds.target is not None
                else np.nan
            )
            runs.append(labels)
            seed_rows.append(
                {
                    "dataset": ds.name,
                    "method": name,
                    "k": k,
                    "seed": seed,
                    "inertia": float(model.inertia_),
                    "n_iter": int(model.n_iter_),
                    "fit_seconds": seconds,
                    "ari": ari,
                }
            )
        rows = pd.DataFrame(
            [
                r
                for r in seed_rows
                if r["dataset"] == ds.name and r["method"] == name
            ]
        )
        ablation.append(
            {
                "ablation": "init",
                "dataset": ds.name,
                "method": name,
                "k": k,
                "inertia_mean": rows.inertia.mean(),
                "inertia_std": rows.inertia.std(ddof=0),
                "inertia_min": rows.inertia.min(),
                "inertia_cv_pct": 100
                * rows.inertia.std(ddof=0)
                / rows.inertia.mean(),
                "iter_mean": rows.n_iter.mean(),
                "iter_std": rows.n_iter.std(ddof=0),
                "fit_seconds_mean": rows.fit_seconds.mean(),
                "ari_mean": rows.ari.mean(),
                "ari_std": rows.ari.std(ddof=0),
                "ari_min": rows.ari.min(),
                "ari_max": rows.ari.max(),
                "pairwise_ari_mean": float(pairwise_ari(runs).mean()),
            }
        )

pd.DataFrame(seed_rows).to_csv(out / "init_seeds.csv", index=False)
pd.DataFrame([row for row in ablation if row["ablation"] == "init"]).round(3)

## 5. Ablation B: distance metric on coordinates

In [ ]:
for ds in [d for d in datasets if is_spatial(d)]:
    p = chosen["dbscan"][ds.name]["rule"]
    eps_km, ms = eps_in_km(ds, p["eps"]), p["min_samples"]
    variants = [
        ("haversine", ds.coords_rad, eps_km / EARTH_RADIUS_KM, "haversine"),
        ("euclidean_degrees", ds.coords_deg, eps_km / 111.32, "euclidean"),
        ("euclidean_km", ds.data, p["eps"], "euclidean"),
    ]
    for variant, Z, eps, metric in variants:
        labels, seconds = methods.fit_predict_timed(
            methods.dbscan(eps, ms, metric=metric), Z
        )
        ablation.append(
            {
                "ablation": "metric",
                "dataset": ds.name,
                "variant": variant,
                "eps": eps,
                "eps_km": eps_km,
                "min_samples": ms,
                "fit_seconds": seconds,
                **evaluate(ds.data, labels, seed=SEED),
            }
        )

pd.DataFrame([row for row in ablation if row["ablation"] == "metric"]).round(3)

## 6. Ablation C: feature scaling (Ecoli)

In [ ]:
ecoli = next(d for d in datasets if d.name == "ecoli")
k_ecoli = chosen["k"]["ecoli"]["selected"]

def best_ari_dbscan(X, y, ms=5):
    """DBSCAN at the best-ARI eps on a fine grid around the knee: its best possible showing."""
    sel = tuning.select_dbscan_by_shared_rule(
        X, min_samples_grid=[ms], eps_factors=np.linspace(0.10, 2.5, 30), y_true=y
    )
    params = sel.best_ari
    return params["eps"], evaluate(
        X, methods.dbscan(**params).fit_predict(X), y, seed=SEED
    )

raw = load_dataset("ecoli", scale=False).data
for variant, X in [("raw", raw), ("standardized", ecoli.data)]:
    labels = methods.kmeans(k_ecoli, random_state=SEED).fit_predict(X)
    ablation.append(
        {
            "ablation": "scaling",
            "dataset": "ecoli",
            "variant": variant,
            "method": "K-Means",
            **evaluate(X, labels, ecoli.target, seed=SEED),
        }
    )
    eps, m = best_ari_dbscan(X, ecoli.target)
    ablation.append(
        {
            "ablation": "scaling",
            "dataset": "ecoli",
            "variant": variant,
            "method": "DBSCAN",
            "eps": eps,
            **m,
        }
    )

pd.DataFrame([row for row in ablation if row["ablation"] == "scaling"]).round(3)

## 7. Ablation D: dimensionality (Ecoli, PCA)

Writes `ablation.csv` (all four ablations).

In [ ]:
for dims in [2, 3, 4, 5, 7]:
    X = (
        PCA(n_components=dims, random_state=SEED).fit_transform(ecoli.data)
        if dims < 7
        else ecoli.data
    )
    grid = np.linspace(0.2, 4, 40)
    sel = tuning.select_dbscan_by_shared_rule(
        X, min_samples_grid=[5], eps_factors=grid, y_true=ecoli.target
    )
    viable = sum(
        t["n_clusters"] >= 2 and t["noise_fraction"] < 0.5 for t in sel.tried
    ) / len(grid)
    eps, m = best_ari_dbscan(X, ecoli.target)
    km = evaluate(
        X,
        methods.kmeans(k_ecoli, random_state=SEED).fit_predict(X),
        ecoli.target,
        seed=SEED,
    )
    ablation.append(
        {
            "ablation": "dimensionality",
            "dataset": "ecoli",
            "variant": f"{dims}D",
            "dims": dims,
            "eps_knee": tuning.find_knee(tuning.k_distance_curve(X, 5))[1],
            "eps_best": eps,
            "viable_eps_frac": viable,
            "dbscan_ari": m["ari"],
            "dbscan_noise": m["noise_fraction"],
            "dbscan_clusters": m["n_clusters"],
            "kmeans_ari": km["ari"],
        }
    )

pd.DataFrame(ablation).to_csv(out / "ablation.csv", index=False)
pd.DataFrame([row for row in ablation if row["ablation"] == "dimensionality"]).round(3)

## 8. Fake-data check

Writes `fake_data_check.csv`.

In [ ]:
for ds in datasets:
    density_null = "uniform_box" if is_spatial(ds) else "shuffled"
    k = chosen["k"][ds.name]
    runs = [
        (
            "K-Means",
            f"k={k['silhouette']} (silhouette)",
            "shuffled",
            lambda Z, k=k["silhouette"]: methods.kmeans(
                k, random_state=SEED
            ).fit_predict(Z),
        ),
        (
            "K-Means",
            f"k={k['selected']} (selected)",
            "shuffled",
            lambda Z, k=k["selected"]: methods.kmeans(
                k, random_state=SEED
            ).fit_predict(Z),
        ),
        (
            "DBSCAN",
            json.dumps(chosen["dbscan"][ds.name]["rule"]),
            density_null,
            lambda Z, p=chosen["dbscan"][ds.name]["rule"]: methods.dbscan(
                **p
            ).fit_predict(Z),
        ),
        (
            "HDBSCAN",
            json.dumps(chosen["hdbscan"][ds.name]["rule"]),
            density_null,
            lambda Z, p=chosen["hdbscan"][ds.name]["rule"]: methods.hdbscan(
                **p
            ).fit_predict(Z),
        ),
    ]
    for method, setting, null, fit in runs:
        fake_rows.append(
            {
                "dataset": ds.name,
                "method": method,
                "setting": setting,
                **null_gap(
                    ds.data,
                    fit,
                    null=null,
                    n_nulls=N_FAKE,
                    random_state=SEED,
                    y_true=ds.target,
                ),
            }
        )

pd.DataFrame(fake_rows).to_csv(out / "fake_data_check.csv", index=False)
pd.DataFrame(fake_rows).round(3)

## 9. Shared-table rows for Uber and Ecoli

The report's shared columns (Section 3.3) at the settings the rule chose, and on labelled Ecoli at the
best-ARI settings too, so the cost of choosing without labels is visible. Each setting is refitted for
its fit time and fake-data silhouette (mean of 3). Reads the settings as saved in section 2, so this
section can be rerun on its own. Writes `shared_rule.csv`.

In [ ]:
out = results_dir("hotspot")
chosen = json.loads((out / "chosen_params.json").read_text())
rows = []
for ds in [load_dataset("uber_pickups"), load_dataset("ecoli")]:
    spatial = "coords_deg" in ds
    k = chosen["k"][ds.name]
    picks = [
        (m, "rule", {"n_clusters": k["selected"]})
        for m in ("K-Means (random init)", "K-Means")
    ]
    if ds.target is not None:
        picks.append(("K-Means", "best_ari", {"n_clusters": k["best_ari"]}))
    for method, key in [("DBSCAN", "dbscan"), ("HDBSCAN", "hdbscan")]:
        picks.append((method, "rule", chosen[key][ds.name]["rule"]))
        if ds.target is not None:
            picks.append((method, "best_ari", chosen[key][ds.name]["best_ari"]))

    for method, choice, p in picks:
        fit = fitter(method, p)
        start = time.perf_counter()
        labels = fit(ds.data)
        seconds = time.perf_counter() - start
        m = evaluate(ds.data, labels, ds.target, seed=SEED)
        null = (
            "uniform_box"
            if spatial and method in ("DBSCAN", "HDBSCAN")
            else "shuffled"
        )
        gap = null_gap(ds.data, fit, null=null, n_nulls=N_FAKE, random_state=SEED)
        rows.append(
            {
                "dataset": ds.name,
                "method": method,
                "choice": choice,
                "params": json.dumps(p),
                **{c: m.get(c) for c in COLUMNS},
                "fit_seconds": seconds,
                "null_silhouette": gap["null_silhouette"],
                "null": null,
            }
        )

table = pd.DataFrame(rows)
table.to_csv(out / "shared_rule.csv", index=False)
e = table[(table.dataset == "ecoli") & (table.choice == "rule")]
print(
    "Ecoli, rule choice: Spearman(silhouette, ARI) =",
    round(spearmanr(e.silhouette, e.ari)[0], 3),
)
table.drop(columns=["null"]).round(3)

## 10. Case statistics

The numbers behind the Uber (D3) and Ecoli (D4) case paragraphs: k-distance spread, share of pickups in
lower and midtown Manhattan, the clusters each setting returns; Ecoli class sizes and how much of each
class each case keeps; Ablation A gaps to the best seed. Writes `case_stats.csv`.

In [ ]:
out = results_dir("hotspot")
chosen = json.loads((out / "chosen_params.json").read_text())
curves = json.loads((out / "kdist_curves.json").read_text())
rows = []

def add(dataset, setting, item, value):
    rows.append(
        {"dataset": dataset, "setting": setting, "item": item, "value": value}
    )

uber = load_dataset("uber_pickups")
X, deg = uber.data, uber.coords_deg
kd = chosen["kdist"]["uber_pickups"]
for q in (5, 50, 95):
    add(
        "uber_pickups",
        f"k-distance, min_samples {kd['min_samples']}",
        f"p{q}_km",
        float(np.percentile(curves["uber_pickups"], q)),
    )
(lat_lo, lat_hi), (lon_lo, lon_hi) = MIDTOWN_BOX["lat"], MIDTOWN_BOX["lon"]
box = (
    (deg[:, 0] > lat_lo)
    & (deg[:, 0] < lat_hi)
    & (deg[:, 1] > lon_lo)
    & (deg[:, 1] < lon_hi)
)
add("uber_pickups", "data", "share_in_midtown_box", float(box.mean()))

db, hdb, k = (
    chosen["dbscan"]["uber_pickups"]["rule"],
    chosen["hdbscan"]["uber_pickups"]["rule"],
    chosen["k"]["uber_pickups"]["selected"],
)
settings = {
    f"DBSCAN knee (eps {kd['knee']:.3f} km, min_samples {kd['min_samples']})": methods.dbscan(
        kd["knee"], kd["min_samples"]
    ),
    f"DBSCAN rule (eps {db['eps']:.3f} km, min_samples {db['min_samples']})": methods.dbscan(
        **db
    ),
    f"HDBSCAN rule (mcs {hdb['min_cluster_size']}, min_samples {hdb['min_samples']})": methods.hdbscan(
        **hdb
    ),
    f"K-Means K={k}": methods.kmeans(k, random_state=SEED),
}
for name, model in settings.items():
    labels = model.fit_predict(X)
    add(
        "uber_pickups",
        name,
        "noise_fraction",
        float((labels == NOISE_LABEL).mean()),
    )
    add(
        "uber_pickups",
        name,
        "n_clusters",
        int(len(np.unique(labels[labels != NOISE_LABEL]))),
    )
    for c in np.unique(labels[labels != NOISE_LABEL]):
        members = labels == c
        lat, lon = deg[members].mean(0)
        near = [
            a
            for a, (pa, po) in AIRPORTS.items()
            if abs(lat - pa) < 0.02 and abs(lon - po) < 0.03
        ]
        add(
            "uber_pickups",
            name,
            f"cluster {c} ({lat:.3f}, {lon:.3f}){' ' + near[0] if near else ''}",
            float(members.mean()),
        )

ecoli = load_dataset("ecoli")
X, y = ecoli.data, ecoli.target
sizes = np.bincount(y)
for c, n in enumerate(sizes):
    add("ecoli", "data", f"class {c} ({ecoli.target_names[c]}) size", int(n))
hdb_best, k_best = (
    chosen["hdbscan"]["ecoli"]["best_ari"],
    chosen["k"]["ecoli"]["best_ari"],
)
cases = {
    f"HDBSCAN best-ARI (mcs {hdb_best['min_cluster_size']}, min_samples {hdb_best['min_samples']})": methods.hdbscan(
        **hdb_best
    ).fit_predict(X),
    f"K-Means K={k_best} (best ARI)": methods.kmeans(
        k_best, random_state=SEED
    ).fit_predict(X),
}
for name, labels in cases.items():
    for c, n in enumerate(sizes):
        members = y == c
        add(
            "ecoli",
            name,
            f"class {c} (n={n}) noise_fraction",
            float((labels[members] == NOISE_LABEL).mean()),
        )
        kept = labels[members][labels[members] != NOISE_LABEL]
        add(
            "ecoli",
            name,
            f"class {c} (n={n}) share in its main cluster",
            float(np.bincount(kept).max() / n) if len(kept) else 0.0,
        )

seeds = pd.read_csv(out / "init_seeds.csv")
for name in ["uber_pickups", "ecoli"]:
    s = seeds[seeds.dataset == name]
    best = s.inertia.min()
    for method, g in s.groupby("method"):
        setting = f"Ablation A, {method}, K={int(g.k.iloc[0])}"
        gap = 100 * (g.inertia / best - 1)
        add(name, setting, "gap_to_best_mean_pct", float(gap.mean()))
        add(name, setting, "gap_to_best_max_pct", float(gap.max()))
        if g.ari.notna().any():
            add(name, setting, "seeds_with_ari_ge_0.7", int((g.ari >= 0.7).sum()))

table = pd.DataFrame(rows)
table.to_csv(out / "case_stats.csv", index=False)
table

## 11. Figures

Every hotspot figure, shown and saved to `results/hotspot/figures/`.

In [ ]:
from sc4020.generate_figures import hotspot_figures
hotspot_figures(show=True)